In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import simlib.generate_sims as gen_sims
import simlib.generate_timeseries as gen_times

In [ ]:
# This creates Signal percent change values from -10% to 10% (-0.1 to 0.1)
# and shows/tests whether the key properties of the simulation work as expected.

n_prop = 7 # number of different proportions of delta_s0:delta_r2s to test
# 0 means all change is in r2s and 1 means all change is in s0
proportion_s0_r2s = np.linspace(0,1,n_prop) 
n_s = 21 # number of different percent change values to test
pchange = np.linspace(-0.1, 0.1, n_s)

# Monoexponential is the full exponential decay model
# Taylor1_approx is the first order taylor approximation of the monoexponential model
# Linear_approx drops one interaction term from the taylor approximation,
# which hopefully only makes a small difference in the approximation of the monoexponential model.
decay_models = ["monoexponential", "taylor1_approx", "linear_approx"]
print(f"{pchange=}, {proportion_s0_r2s=}")

inputted_s = np.tile(pchange, (n_prop, 1))

# Calculating with a range of baseline values
# Intentionally using a diff number for each so it's harder to confuse index dimensions
s0_mean_list = [10, 100, 1000, 10000]
r2s_mean_list = [1/20, 1/28, 1/40]
te_list = [14, 28, 42, 56, 70]
te_baseline = 28

delta_s0_p10_sim = np.full((n_prop, n_s, len(s0_mean_list), len(r2s_mean_list)), np.nan)
delta_r2s_p10_sim = np.full((n_prop, n_s, len(s0_mean_list), len(r2s_mean_list)), np.nan)
s_p10_sim = {
    'monoexponential': np.full((n_prop, n_s, len(s0_mean_list), len(r2s_mean_list), len(te_list)), np.nan),
    'taylor1_approx': np.full((n_prop, n_s, len(s0_mean_list), len(r2s_mean_list), len(te_list)), np.nan),
    'linear_approx': np.full((n_prop, n_s, len(s0_mean_list), len(r2s_mean_list), len(te_list)), np.nan)
}

for s0_mean_idx, s0_mean in enumerate(s0_mean_list):
    for r2s_mean_idx, r2s_mean in enumerate(r2s_mean_list):
        for te_write_idx, te_write in enumerate(te_list):
            delta_s0_p10_sim[:,:,s0_mean_idx, r2s_mean_idx], delta_r2s_p10_sim[:,:,s0_mean_idx, r2s_mean_idx] = gen_sims.calc_delta_r2s_s0_given_s_pchange_proportion(
                    inputted_s=inputted_s,
                    s0_mean=s0_mean,
                    r2s_mean=r2s_mean,
                    proportion_s0_r2s=proportion_s0_r2s,
                    te_baseline=te_baseline,
                )

            s_tmp = gen_sims.all_decay_models(
                tes=te_write,
                mean_s0=s0_mean,
                mean_r2star=r2s_mean,
                delta_s0=delta_s0_p10_sim[:,:,s0_mean_idx, r2s_mean_idx],
                delta_r2star=delta_r2s_p10_sim[:,:,s0_mean_idx, r2s_mean_idx]
            )
            s_p10_sim['monoexponential'][:,:,s0_mean_idx, r2s_mean_idx, te_write_idx]= s_tmp['monoexponential']
            s_p10_sim['taylor1_approx'][:,:,s0_mean_idx, r2s_mean_idx, te_write_idx]= s_tmp['taylor1_approx']
            s_p10_sim['linear_approx'][:,:,s0_mean_idx, r2s_mean_idx, te_write_idx]= s_tmp['linear_approx']
# print(f"{s_p10_sim=}")


In [ ]:
# Sanity check that the delta_s0 and delta_r2s are doing what we expect
s0_mean_idx=0
r2s_mean_idx=0
show_delta_s0 = delta_s0_p10_sim[:,:,s0_mean_idx, r2s_mean_idx]
show_delta_r2s = delta_r2s_p10_sim[:,:,s0_mean_idx, r2s_mean_idx]

plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.plot(pchange,(show_delta_s0/s0_mean_list[s0_mean_idx]).T)
plt.title("Delta_S0 values")
plt.subplot(1,3,2)
plt.plot(pchange,(-te_baseline*show_delta_r2s).T)
plt.title("Delta_R2s values")
plt.xlabel("Inputted percent change values (pchange)")
plt.subplot(1,3,3)
plt.plot(pchange,(show_delta_s0/s0_mean_list[s0_mean_idx]-te_baseline*show_delta_r2s).T)
plt.title("Combined Delta values (should match pchange)")

plt.legend(np.round(proportion_s0_r2s, decimals=2), title="Proportion s0 vs r2s")

In [ ]:
# This is showing the percent modeling error of the taylor and linear approximations 
# compared to the monoexponential model,
# across a range of proportions of delta_s0:delta_r2s,
# a range of inputted percent change values, and a range of baseline s0 and r2s values.

# The measurement error from the monoexponential decay model is biggest when there is a larger proportion of delta_r2s
# and when the TE is larger
# Going from Taylor Approx to Linear Approx has the largest added approx error
# when there is a mix (0.5) of delta_s0 and delta_r2s, but as the
# mix approaches 1, the total approx error drops

plt.figure(figsize=(20,40))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    for te_write_idx, te_write in enumerate(te_list):
        plt.subplot(n_prop, len(te_list), te_write_idx+(prop_idx)*len(te_list)+1)
        plt.title(f"Prop S0:R2s {proportion_s0_r2s[prop_idx]:.2f}, TE={te_write}ms")
        for s0_mean_idx, s0_mean in enumerate(s0_mean_list):
            for r2s_mean_idx, r2s_mean in enumerate(r2s_mean_list):
                for midx, model in enumerate(decay_models):
                    # plt.subplot(2, n_prop, pltidx+1)
                    # plt.plot(100*pchange, s[model][pltidx,:], label=model)
                    if model == "monoexponential":
                        mono_exp_vals = np.squeeze(s_p10_sim[model][prop_idx,:,s0_mean_idx, r2s_mean_idx, te_write_idx])
                    else:
                        plt.plot(100*pchange, 100*(np.squeeze(s_p10_sim[model][prop_idx,:,s0_mean_idx, r2s_mean_idx, te_write_idx]) - mono_exp_vals)/mono_exp_vals, linestyle="dashed", label=model)
                        if te_write_idx==0:
                            plt.ylabel("% change from monoexponential")
        plt.ylim(-4,0)
        plt.xlabel("Signal %change")
plt.legend()
plt.show()

s0_mean_idx=-1
r2s_mean_idx=-1
te_write_idx=-1
plt.figure(figsize=(10,25))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(n_prop,1,prop_idx+1)
    for midx, model in enumerate(decay_models):
        plt.plot(pchange,(s_p10_sim[model][prop_idx,:,s0_mean_idx, r2s_mean_idx, te_write_idx]).T, label=model)
    plt.title(f"S0_mean={s0_mean_list[s0_mean_idx]:.2f}, R2s_mean={r2s_mean_list[r2s_mean_idx]:.2f}, TE={te_list[te_write_idx]}ms, Prop S0:R2s {prop:.2f}")
plt.xlabel("Signal %change")
plt.ylabel("Raw Signal")
plt.legend()
plt.show()


In [ ]:
s0_mean = 500
r2s_mean = 1/40
n_timepoints = 300
n_prop=6
proportion_s0_r2s = np.linspace(0,1,n_prop)
te_baseline = 28
tes = [18, 28, 38, 48, 58]

sim_time = gen_times.gen_bandpass_randn_timeseries(n_reps=1, n_timepoints=n_timepoints, seed=0, passband = [1 / 100, 1 / 20], fs = 0.5,)
sim_time = np.squeeze(0.1 * sim_time / np.max(np.abs(sim_time)))

delta_s0_prop = np.full((n_timepoints, n_prop), np.nan)
delta_r2s_prop = np.full((n_timepoints, n_prop), np.nan)

sim_echoes = np.full((n_timepoints, len(tes), n_prop), np.nan)

for prop_idx, prop in enumerate(proportion_s0_r2s):
    delta_s0_prop[:, prop_idx], delta_r2s_prop[:, prop_idx] = gen_sims.calc_delta_r2s_s0_given_s_pchange_proportion(
                    inputted_s=sim_time,
                    s0_mean=s0_mean,
                    r2s_mean=r2s_mean,
                    proportion_s0_r2s=prop,
                    te_baseline=te_baseline,
                    prop_to_scale="signal"
            )


    for te_idx, te_write in enumerate(tes):
        sim_echoes[:, te_idx, prop_idx] = gen_sims.monoexponential(
                        tes=te_write, #te_baseline,
                        s0=s0_mean + delta_s0_prop[:, prop_idx],
                        r2star=r2s_mean + delta_r2s_prop[:, prop_idx],
                    )

In [ ]:
scaled_sim_echoes = np.full(sim_echoes.shape, np.nan)

plt.figure(figsize=(15,15))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(3,2, prop_idx+1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}")
    plt.plot(sim_echoes[:,:, prop_idx])
    plt.ylabel("Raw Signal")
    plt.xlabel("Timepoints")
plt.show()

plt.figure(figsize=(15,15))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(3,2, prop_idx+1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}")
    scaled_sim_echoes[:,:,prop_idx] = sim_echoes[:,:, prop_idx]/np.mean(sim_echoes[:,:, prop_idx], axis=0, keepdims=True)
    plt.plot(scaled_sim_echoes[:,:, prop_idx])
    plt.ylabel("Mean-Scaled Signal")
    plt.xlabel("Timepoints")
plt.show()

In [ ]:
# Does the signal change when TE == baseline TE?
plt.figure(figsize=(15,15))
te_idx = tes.index(te_baseline)
plt.subplot(2, 1, 1)
plt.plot(np.squeeze(sim_echoes[:,te_idx,:]))
plt.ylabel("Raw Signal")
plt.xlabel("Timepoints")
plt.title(f"TE={te_baseline}ms, should not see signal change across proportions")
plt.subplot(2,1,2)
plt.plot(np.squeeze(scaled_sim_echoes[:,te_idx,:]))
plt.ylabel("Mean-Scaled Signal")
plt.xlabel("Timepoints")
plt.legend(proportion_s0_r2s, title="Proportion s0/r2s")

In [ ]:
plt.figure(figsize=(15,15))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(3,2, prop_idx+1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}, mean scaled")
    plt.plot(tes, sim_echoes[:,:, prop_idx].T)
    plt.ylabel("Raw Signal")
    plt.xlabel("Echoe Time")
plt.show()

plt.figure(figsize=(15,15))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(3,2, prop_idx+1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}, mean scaled")
    plt.plot(tes, scaled_sim_echoes[:,:, prop_idx].T)
    plt.ylabel("Mean-Scaled Signal")
    plt.xlabel("Echoe Time")
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    max_idx = np.argmax(sim_echoes[:, 0, prop_idx])
    #plt.subplot(3, 2, prop_idx + 1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}, max at t0")
    plt.plot(tes, sim_echoes[max_idx, :, prop_idx], label=f"{prop:.2f}")
    plt.ylabel("Raw Signal")
    plt.xlabel("Echo Time")
    plt.legend()
plt.show()

plt.figure(figsize=(5, 5))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    min_idx = np.argmin(sim_echoes[:, 0, prop_idx])
    plt.title(f"Proportion s0/r2s: {prop:.2f}, min at t0")
    #plt.subplot(3, 2, prop_idx + 1)
    plt.plot(tes, sim_echoes[min_idx, :, prop_idx], label=np.round(prop, decimals=2))
    plt.ylabel("Raw Signal")
    plt.xlabel("Echo Time")
    plt.legend()
plt.show()